In [ ]:
import tensorflow as tf
import tensorflow.keras as keras
from keras import layers
import deeplake as dl
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

In [15]:
# Loading the Dataset
train_ds = dl.load("hub://activeloop/visdrone-det-train")
val_ds = dl.load("hub://activeloop/visdrone-det-val")
test_ds = dl.load("hub://activeloop/visdrone-det-test-dev")
print(train_ds.summary())

/

Opening dataset in read-only mode as you don't have write permissions.


/

This dataset can be visualized in Jupyter Notebook by ds.visualize() or at https://app.activeloop.ai/activeloop/visdrone-det-train



/

...

Opening dataset in read-only mode as you don't have write permissions.


-

This dataset can be visualized in Jupyter Notebook by ds.visualize() or at https://app.activeloop.ai/activeloop/visdrone-det-test-dev



-

hub://activeloop/visdrone-det-test-dev loaded successfully.

Dataset(path='hub://activeloop/visdrone-det-train', read_only=True, tensors=['boxes', 'images', 'labels'])

 tensor      htype                 shape               dtype  compression
 -------    -------               -------             -------  ------- 
  boxes      bbox            (6471, 1:914, 4)         float32   None   
 images      image     (6471, 360:1500, 480:2000, 3)   uint8    jpeg   
 labels   class_label          (6471, 1:914)          uint32    None   
None


In [16]:
# Variables
BATCH_SIZE = 32
BATCH_SIZE_CLUSTER = 64
AUTOTUNE = tf.data.AUTOTUNE
pca_model = None
kmeans_model = None
feature_extractor = None
city_labels = ["tianjin", "hong-kong", "daqing", "ganzhou", 
               "guangzhou", "jinchang", "liuzhou", "nanjing", 
               "shaoxing", "shenyang", "nanyang", "zhangjiakou", 
               "suzhou", "xuzhou"]

In [22]:
# 1. PREPROCESSING WORKFLOWS

def image_preprocessing(image):
    image = tf.image.resize(image, (224, 224))
    image = keras.applications.mobilenet_v2.preprocess_input(image)
    return image

def detection_preprocessing(data):
    # Extracts the image and calculates separate counts for humans and cars.
    image = data['images']
    labels = data['labels']  # Integer Class IDs from VisDrone (0 to 9)
    
    # 1. Process the image
    image = image_preprocessing(image)
    
    # 2. Track Humans: Pedestrians (0) OR People (1)
    is_human_mask = tf.logical_or(tf.equal(labels, 0), tf.equal(labels, 1))
    human_count = tf.reduce_sum(tf.cast(is_human_mask, tf.float32))
    
    # 3. Track Cars: Car (3)
    is_car_mask = tf.equal(labels, 3)
    car_count = tf.reduce_sum(tf.cast(is_car_mask, tf.float32))
    
    # 4. Combine into a single multi-count vector: [human_count, car_count]
    # Squeezing elements into a 1D tensor of shape [2]
    counts_vector = tf.stack([human_count, car_count], axis=0)
    
    return image, counts_vector

# 2. FEATURE STREAM PIPELINES (Yields: Image Batch, Count Batch)

train_stream = (train_ds.tensorflow()
                 .map(detection_preprocessing, num_parallel_calls=AUTOTUNE)
                 .batch(BATCH_SIZE_CLUSTER)
                 .prefetch(AUTOTUNE))

val_stream = (val_ds.tensorflow()
                .map(detection_preprocessing, num_parallel_calls=AUTOTUNE)
                .batch(BATCH_SIZE_CLUSTER)
                .prefetch(AUTOTUNE))

test_stream = (test_ds.tensorflow()
                 .map(detection_preprocessing, num_parallel_calls=AUTOTUNE)
                 .batch(BATCH_SIZE_CLUSTER)
                 .prefetch(AUTOTUNE))

In [23]:
# Processing City Labels
def classification_preprocessing(data_stream, is_training=False):
    global pca_model, kmeans_model, feature_extractor

    if feature_extractor is None:
        feature_extractor = keras.applications.MobileNetV2(
            weights='imagenet', include_top=False, pooling='avg', input_shape=(224, 224, 3)
        )
    features = feature_extractor.predict(data_stream, verbose=1)
    
    if is_training:
        pca_model = PCA(n_components=50, random_state=42)
        kmeans_model = KMeans(n_clusters=14, random_state=42, n_init=10)
        
        reduced_features = pca_model.fit_transform(features)
        labels = kmeans_model.fit_predict(reduced_features)
    else:
        reduced_features = pca_model.transform(features)
        labels = kmeans_model.predict(reduced_features)
        
    return tf.constant(labels, dtype=tf.int32)

tf_train_labels = classification_preprocessing(train_stream.map(lambda img, _ : img), is_training=True)
tf_val_labels = classification_preprocessing(val_stream.map(lambda img, _ : img), is_training=False)
tf_test_labels = classification_preprocessing(test_stream.map(lambda img, _ : img), is_training=False)

102/102 ━━━━━━━━━━━━━━━━━━━━ 218s 2s/step


c:\Users\etito\miniconda3\Lib\site-packages\keras\src\trainers\epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


9/9 ━━━━━━━━━━━━━━━━━━━━ 16s 2s/step


c:\Users\etito\miniconda3\Lib\site-packages\keras\src\trainers\epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


26/26 ━━━━━━━━━━━━━━━━━━━━ 50s 2s/step


c:\Users\etito\miniconda3\Lib\site-packages\keras\src\trainers\epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


In [24]:
# Adding Pipelines

train_labels_ds = tf.data.Dataset.from_tensor_slices(tf_train_labels).batch(BATCH_SIZE_CLUSTER)
val_labels_ds = tf.data.Dataset.from_tensor_slices(tf_val_labels).batch(BATCH_SIZE_CLUSTER)
test_labels_ds = tf.data.Dataset.from_tensor_slices(tf_test_labels).batch(BATCH_SIZE_CLUSTER)

def shape_structure_outputs(img_batch, count_batch, label_batch):
    """
    Enforces static shapes for the Keras graph and structures targets 
    to match the multi-head model's output names.
    """
    # Enforce static ranks for the Keras graph builder
    img_batch.set_shape([None, 224, 224, 3])
    count_batch.set_shape([None, 2])
    label_batch.set_shape([None])   # [Batch_Size]
    
    return img_batch, {
        "city_output": label_batch,
        "count_output": count_batch     # Changed key name to match your model's counting head
    }

# Zip your feature streams (images + counts) with your clustering labels
train_pipeline = tf.data.Dataset.zip((train_stream, train_labels_ds)).map(
    lambda stream_data, labels: shape_structure_outputs(stream_data[0], stream_data[1], labels),
    num_parallel_calls=AUTOTUNE
).prefetch(AUTOTUNE)

val_pipeline = tf.data.Dataset.zip((val_stream, val_labels_ds)).map(
    lambda stream_data, labels: shape_structure_outputs(stream_data[0], stream_data[1], labels),
    num_parallel_calls=AUTOTUNE
).prefetch(AUTOTUNE)

test_pipeline = tf.data.Dataset.zip((test_stream, test_labels_ds)).map(
    lambda stream_data, labels: shape_structure_outputs(stream_data[0], stream_data[1], labels),
    num_parallel_calls=AUTOTUNE
).prefetch(AUTOTUNE)

In [30]:
# Building the Model
def build_multi_head_model():
    input_tensor = layers.Input(shape=(224, 224, 3), name='input_image')

    base_model = keras.applications.MobileNetV2(
        weights='imagenet', 
        include_top=False, 
        input_tensor=input_tensor
    )
    base_model.trainable = False

    shared_features = base_model.output

    # 1. Object Detection Head (Processes backbone features first)
    x_detect = layers.GlobalAveragePooling2D()(shared_features)
    x_detect = layers.Dense(128, activation='relu')(x_detect)
    count_output = layers.Dense(2, activation='sigmoid', name="count_output")(x_detect)

    # 2. City Classification Head (Dependent on Detection)
    x_class_input = layers.GlobalAveragePooling2D()(shared_features)
    
    # Concatenate blends the spatial context with the localization features
    merged_features = layers.Concatenate()([x_class_input, x_detect])
    
    x_class = layers.Dense(256, activation='relu')(merged_features)
    x_class = layers.Dropout(0.5)(x_class)
    city_output = layers.Dense(14, activation='softmax', name="city_output")(x_class)

    model = keras.Model(inputs=input_tensor, outputs=[city_output, count_output])

    lr_scheduler = keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=0.001,
        decay_steps=10000,
        alpha=0.0
    )
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr_scheduler),
        loss={
            "city_output": "sparse_categorical_crossentropy",
            "count_output": "mean_squared_error"
        },
        metrics={
            "city_output": "accuracy",
            "count_output": "mae"
        },
        loss_weights={
            "city_output": 1.0, 
            "count_output": 2.0
        }
    )
    model.fit(
        train_pipeline,
        validation_data=val_pipeline,
        epochs=1
    )
    
    return model

china_model = build_multi_head_model()

C:\Users\etito\AppData\Local\Temp\ipykernel_21324\1507852423.py:5: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = keras.applications.MobileNetV2(


    102/Unknown 224s 2s/step - city_output_accuracy: 0.4840 - city_output_loss: 1.6044 - count_output_loss: 340.6060 - count_output_mae: 7.3913 - loss: 682.8638

c:\Users\etito\miniconda3\Lib\site-packages\keras\src\trainers\epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


102/102 ━━━━━━━━━━━━━━━━━━━━ 241s 2s/step - city_output_accuracy: 0.6308 - city_output_loss: 1.0619 - count_output_loss: 331.0848 - count_output_mae: 7.3139 - loss: 674.5659 - val_city_output_accuracy: 0.7828 - val_city_output_loss: 0.5384 - val_count_output_loss: 333.6461 - val_count_output_mae: 9.8767 - val_loss: 760.4822
